In [11]:
import pandas as pd
import re
from pathlib import Path

# Define input/output directories
SOURCE_DIR = Path('datasets/source_datasets')
OUTPUT_DIR = Path('datasets/merged_dataset')

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all .dat files in source directory
dat_files = list(SOURCE_DIR.glob('tij_*.dat'))

if not dat_files:
    print("No .dat files found in source_datasets directory")
    print(f"Please add contact tracing .dat files to: {SOURCE_DIR}")
else:
    print(f"Found {len(dat_files)} dataset(s) to process\n")
    
    for dat_file in dat_files:
        print(f"Processing: {dat_file.name}")
        
        # Read contact data
        contacts = pd.read_csv(dat_file, sep=' ', header=None, 
                              names=['timestamp', 'person_A', 'person_B'])
        
        # Ensure numeric types
        contacts['timestamp'] = pd.to_numeric(contacts['timestamp'], errors='coerce')
        contacts['person_A'] = pd.to_numeric(contacts['person_A'], errors='coerce')
        contacts['person_B'] = pd.to_numeric(contacts['person_B'], errors='coerce')
        contacts = contacts.dropna(subset=['timestamp', 'person_A', 'person_B'])
        
        # Look for corresponding metadata file
        base_name = dat_file.stem.replace('tij_', '')
        metadata_file = SOURCE_DIR / f'metadata_{base_name}.dat'
        
        if metadata_file.exists():
            print(f"  ✓ Found metadata: {metadata_file.name}")
            
            # Read and process metadata
            meta = pd.read_csv(metadata_file, header=None, names=['raw'])
            
            # Extract person IDs and group labels
            meta['person_id'] = meta['raw'].apply(
                lambda x: int(re.findall(r'\d+', str(x))[0]) if re.findall(r'\d+', str(x)) else None
            )
            meta['group'] = meta['raw'].apply(
                lambda x: re.findall(r'[A-Za-z]+', str(x))[0] if re.findall(r'[A-Za-z]+', str(x)) else None
            )
            meta = meta.dropna(subset=['person_id'])
            
            # Merge group labels for both persons
            contacts = contacts.merge(
                meta[['person_id', 'group']], 
                left_on='person_A', 
                right_on='person_id', 
                how='left'
            ).rename(columns={'group': 'group_A'}).drop(columns=['person_id'])
            
            contacts = contacts.merge(
                meta[['person_id', 'group']], 
                left_on='person_B', 
                right_on='person_id', 
                how='left'
            ).rename(columns={'group': 'group_B'}).drop(columns=['person_id'])
        else:
            print(f"No metadata file found, skipping group enrichment")
            contacts['group_A'] = None
            contacts['group_B'] = None
        
        # Standardize timestamps (SocioPatterns use 20s intervals)
        contacts['time_seconds'] = contacts['timestamp'] * 20
        contacts['time_minutes'] = contacts['time_seconds'] / 60
        contacts['time_hours'] = contacts['time_seconds'] / 3600
        
        # Save merged dataset
        output_file = OUTPUT_DIR / f"{dat_file.stem}_merged.csv"
        contacts.to_csv(output_file, index=False)
        print(f"  ✓ Saved: {output_file.name}")
        print(f"  ✓ Rows: {len(contacts):,}\n")
    
    print(f"Processing complete! All files saved to: {OUTPUT_DIR}")

Found 5 dataset(s) to process

Processing: tij_Thiers13.dat
  ✓ Found metadata: metadata_Thiers13.dat
  ✓ Saved: tij_Thiers13_merged.csv
  ✓ Rows: 188,508

Processing: tij_SFHH.dat
  ✓ Found metadata: metadata_SFHH.dat
  ✓ Saved: tij_Thiers13_merged.csv
  ✓ Rows: 188,508

Processing: tij_SFHH.dat
  ✓ Found metadata: metadata_SFHH.dat
  ✓ Saved: tij_SFHH_merged.csv
  ✓ Rows: 70,261

Processing: tij_LH10.dat
  ✓ Found metadata: metadata_LH10.dat
  ✓ Saved: tij_LH10_merged.csv
  ✓ Rows: 0

Processing: tij_InVS15.dat
  ✓ Found metadata: metadata_InVS15.dat
  ✓ Saved: tij_SFHH_merged.csv
  ✓ Rows: 70,261

Processing: tij_LH10.dat
  ✓ Found metadata: metadata_LH10.dat
  ✓ Saved: tij_LH10_merged.csv
  ✓ Rows: 0

Processing: tij_InVS15.dat
  ✓ Found metadata: metadata_InVS15.dat
  ✓ Saved: tij_InVS15_merged.csv
  ✓ Rows: 78,249

Processing: tij_InVS13.dat
  ✓ Found metadata: metadata_InVS13.dat
  ✓ Saved: tij_InVS13_merged.csv
  ✓ Rows: 9,827

Processing complete! All files saved to: datasets/

Compile All Merged Datasets

Combine all individual merged CSV files into one master dataset.

In [12]:
import pandas as pd
import os
from pathlib import Path

# Find all merged CSV files
merged_files = list(Path('datasets/merged_dataset').glob('*.csv'))

if not merged_files:
    print("⚠️ No merged CSV files found in datasets/merged_dataset")
else:
    print(f"Found {len(merged_files)} merged dataset(s) to compile\n")
    
    all_data = []
    
    for csv_file in merged_files:
        print(f"Adding: {csv_file.name}")
        df = pd.read_csv(csv_file)
        df['source_file'] = csv_file.stem
        all_data.append(df)
        print(f"  Rows: {len(df):,}")
    
    # Merge all datasets
    merged_df = pd.concat(all_data, ignore_index=True)
    
    # Save compiled dataset
    output_file = "datasets/compiled_merged_dataset/all_compiled_contacts.csv"
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    merged_df.to_csv(output_file, index=False)
    
    print(f"\n✅ Successfully compiled {len(all_data)} datasets into '{output_file}'")
    print(f"Total rows: {len(merged_df):,}")

Found 6 merged dataset(s) to compile

Adding: tij_SFHH_merged.csv
  Rows: 70,261
Adding: tij_InVS15_merged.csv
  Rows: 78,249
Adding: tij_InVS15.csv
  Rows: 78,249
Adding: tij_Thiers13_merged.csv
  Rows: 188,508
Adding: tij_LH10_merged.csv
  Rows: 188,508
Adding: tij_LH10_merged.csv
  Rows: 0
Adding: tij_InVS13_merged.csv
  Rows: 9,827
  Rows: 0
Adding: tij_InVS13_merged.csv
  Rows: 9,827


/var/folders/4d/c9fb2t_j5znf_ftgytncz11h0000gp/T/ipykernel_86823/2807814732.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat(all_data, ignore_index=True)



✅ Successfully compiled 6 datasets into 'datasets/compiled_merged_dataset/all_compiled_contacts.csv'
Total rows: 425,094


## Step 3: Final Data Preprocessing

Clean and prepare the compiled dataset for analysis.

In [13]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the compiled dataset
df = pd.read_csv("datasets/compiled_merged_dataset/all_compiled_contacts.csv")

print(f"Initial dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Initial dataset shape: (425094, 9)
Columns: ['timestamp', 'person_A', 'person_B', 'group_A', 'group_B', 'time_seconds', 'time_minutes', 'time_hours', 'source_file']


/var/folders/4d/c9fb2t_j5znf_ftgytncz11h0000gp/T/ipykernel_86823/1581915060.py:5: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasets/compiled_merged_dataset/all_compiled_contacts.csv")


In [14]:
# Drop duplicates and nulls
df = df.drop_duplicates()
df = df.dropna(subset=["timestamp", "person_A", "person_B"])

print(f"After removing duplicates and nulls: {df.shape}")

After removing duplicates and nulls: (425094, 9)


In [15]:
# Remove self-contacts
df = df[df["person_A"] != df["person_B"]]

print(f"After removing self-contacts: {df.shape}")

After removing self-contacts: (425094, 9)


In [16]:
# Convert datatypes
df["person_A"] = df["person_A"].astype(int)
df["person_B"] = df["person_B"].astype(int)
df["timestamp"] = df["timestamp"].astype(float)

print("✓ Datatypes converted")

✓ Datatypes converted


In [17]:
# Convert timestamp to real datetime
# timestamps are in seconds from April 17, 2009
start_date = pd.to_datetime("2009-04-17 00:00:00")
df["datetime"] = start_date + pd.to_timedelta(df["timestamp"], unit="s")

# Separate into date and hour columns
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.time

print("✓ Datetime columns created")

✓ Datetime columns created


In [18]:
# Sort chronologically
df = df.sort_values(by="datetime").reset_index(drop=True)

print("✓ Data sorted chronologically")

✓ Data sorted chronologically


In [19]:
# Normalize timestamp (optional, for modeling)
scaler = MinMaxScaler()
df["timestamp_norm"] = scaler.fit_transform(df[["timestamp"]])

print("✓ Timestamp normalized")

✓ Timestamp normalized
